# Aff-Wild2 Stage 2 — audio extraction

Pipeline:
1. Bridge the MTL-release annotations into `data/affwild2/annotations/` so the existing Stage 2/3 configs work as-is.
2. `ffmpeg` over `affwild2/videos/` (598 files across `batch1/`, `batch2/`, `new_vids/`) → `cache/audio_wav/<videoname>.wav` (16 kHz mono).
3. HuBERT-large features over the wavs → `cache/features/hubert_large/<videoname>.npz`.
4. wav2vec 2.0 base features (RQ3 audio-encoder slot) → `cache/features/wav2vec2_base/<videoname>.npz`.

All steps are idempotent: existing outputs are skipped unless `overwrite=True`. Only the 307 MTL-release video stems are consumed downstream by `align_audio_to_video.py`; the extra ~291 unmatched files become inert dead weight (kept for simplicity).

Wall-time on a 3050 Ti: ~30 min wav + ~2–4 h per encoder.

In [1]:
import os, sys
from pathlib import Path

REPO = Path.cwd().resolve().parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

VIDEOS_DIR     = (REPO.parent / 'affwild2' / 'videos').resolve()
WAV_CACHE      = REPO / 'cache' / 'audio_wav'
HUBERT_CACHE   = REPO / 'cache' / 'features' / 'hubert_large'
WAV2VEC2_CACHE = REPO / 'cache' / 'features' / 'wav2vec2_base'

print('cwd        :', Path.cwd())
print('videos_dir :', VIDEOS_DIR, '\u2192 exists?', VIDEOS_DIR.exists())
print('wav cache  :', WAV_CACHE)

cwd        : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code
videos_dir : C:\Users\Andrey Lyaschenko\Documents\vkr\affwild2\videos → exists? True
wav cache  : C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\cache\audio_wav


## 0. Bridge MTL annotations into `data/affwild2/annotations/`

Stage 1 reads annotations from `../competition-data/`; Stage 2/3 configs read from `data/affwild2/annotations/`. Copy once so the existing configs work unmodified.

In [2]:
import shutil

ANN_SRC = (REPO.parent / 'competition-data').resolve()
ANN_DST = REPO / 'data' / 'affwild2' / 'annotations'
ANN_DST.mkdir(parents=True, exist_ok=True)

for name in ('training_set_annotations.txt', 'validation_set_annotations.txt'):
    src, dst = ANN_SRC / name, ANN_DST / name
    if not src.exists():
        raise FileNotFoundError(f'missing source annotation: {src}')
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print(f'[skip] {name} already bridged ({dst.stat().st_size:,} bytes)')
        continue
    shutil.copy2(src, dst)
    print(f'[copy] {src} \u2192 {dst} ({dst.stat().st_size:,} bytes)')

[skip] training_set_annotations.txt already bridged (8,752,473 bytes)
[skip] validation_set_annotations.txt already bridged (1,644,034 bytes)


## 1. ffmpeg sanity check

Both `ffmpeg` and `ffprobe` must be on PATH; `extract_audio_wav.py` shells out to them.

In [3]:
  import shutil, subprocess, os
  from pathlib import Path

  if shutil.which('ffmpeg') is None:
      candidates = []
      home = Path.home()
      for base in (home / 'Downloads', home / 'AppData' / 'Local',
                   Path('C:/ffmpeg'), Path('C:/Program Files/ffmpeg'),
                   Path('C:/Program Files (x86)/ffmpeg')):
          if not base.exists():
              continue
          candidates += list(base.glob('**/bin/ffmpeg.exe'))
      if candidates:
          bin_dir = str(candidates[0].parent)
          os.environ['PATH'] = bin_dir + os.pathsep + os.environ['PATH']
          print(f'[fix] prepended {bin_dir} to PATH')

  for tool in ('ffmpeg', 'ffprobe'):
      path = shutil.which(tool)
      if path is None:
          print(f'[diag] {tool} not found. PATH entries containing "ffmpeg":')
          for entry in os.environ['PATH'].split(os.pathsep):
              if 'ffmpeg' in entry.lower():
                  print(' ', entry)
          print('[diag] `where ffmpeg` from the OS:')
          subprocess.run(['where', 'ffmpeg'], shell=True)
          raise AssertionError(
              f'{tool!r} not on PATH. Either fully restart the kernel after '
              'installing ffmpeg, or set os.environ["PATH"] inline above this cell.'
          )
      out = subprocess.run([tool, '-version'], capture_output=True, text=True)
      print(f'{tool:8s}: {path}')
      print('         ', out.stdout.splitlines()[0])

[fix] prepended C:\Users\Andrey Lyaschenko\Downloads\ffmpeg-2026-04-30-git-cc3ca17127-essentials_build\ffmpeg-2026-04-30-git-cc3ca17127-essentials_build\bin to PATH
ffmpeg  : C:\Users\Andrey Lyaschenko\Downloads\ffmpeg-2026-04-30-git-cc3ca17127-essentials_build\ffmpeg-2026-04-30-git-cc3ca17127-essentials_build\bin\ffmpeg.EXE
          ffmpeg version 2026-04-30-git-cc3ca17127-essentials_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
ffprobe : C:\Users\Andrey Lyaschenko\Downloads\ffmpeg-2026-04-30-git-cc3ca17127-essentials_build\ffmpeg-2026-04-30-git-cc3ca17127-essentials_build\bin\ffprobe.EXE
          ffprobe version 2026-04-30-git-cc3ca17127-essentials_build-www.gyan.dev Copyright (c) 2007-2026 the FFmpeg developers


## 2. Extract 16 kHz mono `.wav`s over `affwild2/videos/`

Walks `batch1/`, `batch2/`, `new_vids/` recursively. Clips without an audio stream get a `<name>.silent` marker and are silently skipped downstream. Wall-time: ~30 min for 598 files.

In [4]:
from src.features.extract_audio_wav import extract_audio_wavs

WAV_CACHE.mkdir(parents=True, exist_ok=True)
written = extract_audio_wavs(
    videos_dir=VIDEOS_DIR,
    output_dir=WAV_CACHE,
    sample_rate=16000,
    channels=1,
    overwrite=False,
)
print(f'wav cache: {len(written)} files @ {WAV_CACHE}')

ffmpeg: 100%|██████████| 598/598 [02:04<00:00,  4.80it/s]

[audio] wrote 598 wav files to C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\cache\audio_wav
wav cache: 598 files @ C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\cache\audio_wav


In [5]:
wavs    = sorted(WAV_CACHE.glob('*.wav'))
silents = sorted(WAV_CACHE.glob('*.silent'))
print(f'wavs   : {len(wavs)}')
print(f'silent : {len(silents)}  (clips with no usable audio)')
if silents:
    print('  sample silent stems:', [p.stem for p in silents[:10]])

# Cross-check coverage of the 307 MTL stems via the visual cache.
ENET_CACHE = REPO / 'cache' / 'features' / 'enet_b0_8_va_mtl'
mtl_stems = {p.stem for p in ENET_CACHE.glob('*.npz')}
wav_stems = {p.stem for p in wavs}
missing_audio_for_mtl = sorted(mtl_stems - wav_stems - {p.stem for p in silents})
print(f'MTL stems missing both wav and silent marker: {len(missing_audio_for_mtl)}')
if missing_audio_for_mtl:
    print('  sample:', missing_audio_for_mtl[:10])

# How many MTL stems have a wav vs silent?
mtl_with_wav    = mtl_stems & wav_stems
mtl_with_silent = mtl_stems & {p.stem for p in silents}
print(f'MTL stems with wav   : {len(mtl_with_wav)} / {len(mtl_stems)}')
print(f'MTL stems silent     : {len(mtl_with_silent)} / {len(mtl_stems)}')

wavs   : 598
silent : 0  (clips with no usable audio)
MTL stems missing both wav and silent marker: 9
  sample: ['10-60-1280x720_right', '135-24-1920x1080_left', '135-24-1920x1080_right', '46-30-484x360_left', '46-30-484x360_right', '6-30-1920x1080_left', '6-30-1920x1080_right', 'video2_left', 'video49_left']
MTL stems with wav   : 298 / 307
MTL stems silent     : 0 / 307


## 3. HuBERT-large features (primary audio backbone)

Output: `cache/features/hubert_large/<videoname>.npz` with `features (T_audio, 1024)`, `hop_sec`, `wav_seconds`.

Wall-time on 3050 Ti: ~2–4 h for 598 wavs at 10-s window / 1-s overlap.

In [7]:
from src.features.extract_audio_features import extract_audio_features

HUBERT_CACHE.mkdir(parents=True, exist_ok=True)
extract_audio_features(
    backbone='hubert_large',
    wav_dir=WAV_CACHE,
    output_dir=HUBERT_CACHE,
    device=None,           # cuda if available, else cpu
    window_sec=10.0,
    overlap_sec=1.0,
    overwrite=False,
)

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 422/422 [00:00<00:00, 74466.59it/s]
HubertModel LOAD REPORT from: facebook/hubert-large-ls960-ft
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
audio[hubert_large]: 100%|██████████| 607/607 [31:18<00:00,  3.09s/it] 


In [8]:
import numpy as np

npz_files = sorted(HUBERT_CACHE.glob('*.npz'))
print(f'hubert_large cache: {len(npz_files)} files')
assert npz_files, 'no hubert_large features written'

sample = np.load(npz_files[0])
print('keys      :', list(sample.keys()))
print('features  :', sample['features'].shape, sample['features'].dtype)
print('hop_sec   :', float(sample['hop_sec']))
print('wav_seconds:', float(sample['wav_seconds']))
assert sample['features'].shape[1] == 1024, 'expected D=1024 for HuBERT-large'

# T_audio histogram across all clips
T = np.array([np.load(p)['features'].shape[0] for p in npz_files])
print(f'T_audio: min={T.min()}  median={int(np.median(T))}  max={T.max()}  mean={T.mean():.0f}')

hubert_large cache: 607 files
keys      : ['features', 'hop_sec', 'wav_seconds']
features  : (17226, 1024) float32
hop_sec   : 0.020039167255163193
wav_seconds: 345.1947021484375
T_audio: min=130  median=7414  max=78892  mean=9162


## 4. wav2vec 2.0 base features (RQ3 audio-encoder ablation)

Lower-dim (768) and lighter than HuBERT-large; serves as the encoder-ablation slot for **RQ3**. Same input wavs, same window/overlap.

In [9]:
WAV2VEC2_CACHE.mkdir(parents=True, exist_ok=True)
extract_audio_features(
    backbone='wav2vec2_base',
    wav_dir=WAV_CACHE,
    output_dir=WAV2VEC2_CACHE,
    device=None,
    window_sec=10.0,
    overlap_sec=1.0,
    overwrite=False,
)

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Andrey Lyaschenko\.cache\huggingface\hub\models--facebook--wav2vec2-base-960h. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 210/210 [00:00<00:00, 9120.70it/s]
Wav2Vec2Model

In [10]:
npz_files = sorted(WAV2VEC2_CACHE.glob('*.npz'))
print(f'wav2vec2_base cache: {len(npz_files)} files')
assert npz_files, 'no wav2vec2_base features written'

sample = np.load(npz_files[0])
print('features  :', sample['features'].shape, sample['features'].dtype)
assert sample['features'].shape[1] == 768, 'expected D=768 for wav2vec2-base'
T = np.array([np.load(p)['features'].shape[0] for p in npz_files])
print(f'T_audio: min={T.min()}  median={int(np.median(T))}  max={T.max()}  mean={T.mean():.0f}')

wav2vec2_base cache: 607 files
features  : (17226, 768) float32
T_audio: min=130  median=7414  max=78892  mean=9162


In [11]:
# 5. Alias audio for _left/_right split tracks.
# These MTL stems share their parent video's audio (e.g. 10-60-1280x720_left
# and _right both come from 10-60-1280x720.mp4). The wav was written under
# the parent stem; align_audio_to_video.py looks up audio by exact stem, so
# without this alias step the 9 split tracks become zero-filled audio.
# Idempotent: re-running is a no-op once aliases exist.
import shutil

ENET_CACHE = REPO / 'cache' / 'features' / 'enet_b0_8_va_mtl'
mtl_stems = sorted(p.stem for p in ENET_CACHE.glob('*.npz'))
split_stems = [s for s in mtl_stems if s.endswith('_left') or s.endswith('_right')]

CACHES_TO_ALIAS = [
    (WAV_CACHE,      '.wav'),
    (HUBERT_CACHE,   '.npz'),
    (WAV2VEC2_CACHE, '.npz'),
]

for cache_dir, ext in CACHES_TO_ALIAS:
    if not cache_dir.exists():
        print(f'[skip] {cache_dir} does not exist yet')
        continue
    n_aliased = 0
    n_skipped = 0
    n_missing_parent = 0
    for stem in split_stems:
        parent = stem.replace('_left', '').replace('_right', '')
        src = cache_dir / f'{parent}{ext}'
        dst = cache_dir / f'{stem}{ext}'
        if dst.exists():
            n_skipped += 1
            continue
        if not src.exists():
            n_missing_parent += 1
            continue
        shutil.copy2(src, dst)
        n_aliased += 1
    print(f'{cache_dir.name:25s}: aliased={n_aliased}  skipped={n_skipped}  missing_parent={n_missing_parent}')

audio_wav                : aliased=0  skipped=9  missing_parent=0
hubert_large             : aliased=0  skipped=9  missing_parent=0
wav2vec2_base            : aliased=0  skipped=9  missing_parent=0


Audio caches are now populated for both encoders. Proceed to `aw2_04_align_audio.ipynb` to resample them onto the visual frame timeline.